In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format

In [3]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
login(token = hf_token)

In [ ]:
wb_token = user_secrets.get_secret("wandb")

wandb.login(key=wb_token)
run = wandb.init(
    project='Llama-instruct-abstract', 
    job_type="training", 
    anonymous="allow"
)

In [ ]:
base_model = "/kaggle/input/llama-3.2/transformers/3b-instruct/1"
new_model = "llama-3.2-3b-abstract-4"

In [ ]:
# Set torch dtype and attention implementation
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    attn_implementation = "eager"

In [ ]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)
# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer))

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files={"/kaggle/input/abstract/wsj_data_300.json"}) #,split='train') 
dataset = dataset.shuffle(seed=65)

In [ ]:
for sample in dataset["test"]:
    type = ['B-DAV', 'I-DAV', 'B-IAV', 'I-IAV','B-SV', 'I-SV', 'B-ADJD', 'I-ADJD', 'B-ADJI', 'I-ADJI','B-NOUN', 'I-NOUN', 'O']
    if len(sample["tokens"]) != len(sample["labels"]):
        print("wrong length", sample["id"])
    for label in sample["labels"]:
        if label not in type:
            print("wrong label", sample["id"], label)

In [ ]:
def format_chat_template(row):
    instruction = """Instruction: Your task is to analyze the input text and perform the token classification task.
        Use the following labels and their meanings to assign label to each token: 
        1. DAV (descriptive action verbs): Verbs or phrases that describe a specific, observable action with a clear beginning and end, and with a common physical characteristic.
            Examples: "hit", "yell", "walk", "read", "discuss", "talk", "ride bike", "video chat"
            Note: If used metaphorically, consider IAV or SV instead of DAV.
        2. IAV (interpretative action verb): Verbs or phrases that describe actions with clear beginning and end but no invariant physical feature. These do not express cognitive or emotional states.
            Examples: "make", "take", "study", "exercise", "help", "tease", "wake up", "follow", "provide", "benefit", "repeat", "change clothes", "relax"
            Note: Auxiliary verbs can be part of an IAV span but cannot independently be an IAV.
        3. SV (state verbs): Verbs or phrases indicating emotional or psychological states, or changes in these states.
            Examples: "surprise", "amaze", "anger", "fear", "love", "appreciate", "think", "understand", "consider", "know", "guess", "attend to"
            Example: In "The experience changed me," "changed" is an SV.
        4. ADJD (adjectives): Words or spans describing abstract characteristics or features, including adjectives, qualifying adverbs and verbs, and noun clauses that are descriptions.
    the sentence has no meaning in and of itself without reference to the noun, and noun clauses that are a decription. 
            Examples: "honest", "aggresive", "unknow", "small", "wrong", "pleasant", "firmly"
            Example: In “I pass people waiting for the train .”, "waiting for the train" is an ADJD.
            Example: "what you eat" in the sentence "You are what you eat ." is an ADJD.
            Note: When a verb lacks standalone meaning, refer to the modifying span (e.g., "religion" in "He had a religion", and "a medal" in "He deserved a medal .").
        5. ADJI (identifying adjectives): refer to a specific type of adjectives which indicate the identity of the described.
            Examples: "Chinese" in "Chinese products", "black" in "black people", "gay" in "gay people", "Democratic"
        6. NOUN (nouns): refer to nouns which indicate the identity (group) of the described, or proper nouns that refer to a political entity.
            Examples: "American" (noun), "America", "Democrats", "woman", "feminists", "environmentalists", "the gays"
        If a token does not fit into any of the defined categories, assign it the label O (Outside).
        Labeling Rules:
        1. Use the BIO scheme:
            B-LABEL: The first token of a labeled span.
            I-LABEL: Subsequent tokens of the same span.
            O: Tokens not belonging to any labeled span.
        2. Label at the token level.
        3. Be consistent with the definitions provided.
        Output Format:
        Ensure your output maintains the labeling scheme and the same token order and as the input text, as shown in the example below.
        Example Input:
            His father is a thief . </s>
        Example Output:
            [His O] [father O] [is O] [a B-ADJD] [thief I-ADJD] [. O] </s>
        """

    row_json = [{"role": "system", "content": instruction },
               {"role": "user", "content": row["text"]+"</s>"}]
    
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(
    format_chat_template,
    num_proc= 4,
)

In [ ]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [ ]:
# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # task_type="TOKEN_CLS",
    target_modules=modules
)
tokenizer.chat_template = None
model, tokenizer = setup_chat_format(model, tokenizer)
model = get_peft_model(model, peft_config)

In [ ]:
training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    group_by_length=True,
    report_to="wandb",
    max_seq_length = 2048,
    packing= False,
)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

#Hyperparamter
training_arguments = SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        warmup_steps = 10,
        num_train_epochs = 15,
        logging_steps=50,
        logging_strategy="steps",
        learning_rate = 2e-4,
        fp16 = True, 
        # bf16 = False, 
        group_by_length=True,
        optim = "paged_adamw_32bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = new_model,
        report_to = "wandb",
        max_seq_length = 1024,
        dataset_num_proc = 4,
        packing = False, 
    )

In [ ]:
class TOKSFTTrainer(SFTTrainer):
    # def __init__(self, *args, **kwargs):
    #     super().__init__(*args, **kwargs)
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Get the device of the first layer's weights as reference
        try:
            # Try to get device from the base model if it exists
            model_device = model.base_model.device
        except AttributeError:
            # Fallback to getting device from classifier weights
            model_device = model.classifier.weight.device
            
        # Explicitly move all input tensors to the same device
        inputs = {k: v.to(model_device) if isinstance(v, torch.Tensor) else v 
                 for k, v in inputs.items()}
        
        # Forward pass
        outputs = model(**inputs)
        
        # Get logits and ensure they're on the correct device
        if isinstance(outputs, dict):
            logits = outputs.get("logits")
        else:
            logits = outputs[0]
        
        logits = logits.to(model_device)
        labels = inputs["labels"].to(model_device)
        
        # Print device information for debugging
        print(f"Logits device: {logits.device}")
        print(f"Labels device: {labels.device}")
        
        # Reshape if needed
        if len(logits.shape) == 3:
            logits = logits.view(-1, logits.shape[-1])
        if len(labels.shape) == 2:
            labels = labels.view(-1)
        
        # Calculate loss with explicit device placement
        active_loss = labels != -100
        if active_loss.sum() > 0:
            active_logits = logits[active_loss].to(model_device)
            active_labels = labels[active_loss].to(model_device)
            loss = F.cross_entropy(active_logits, active_labels)
        else:
            loss = torch.tensor(0.0).to(model_device)
        
        return (loss, outputs) if return_outputs else loss


In [ ]:
# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_arguments,
    peft_config=peft_config,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
wandb.finish()
# Save the fine-tuned model
trainer.model.save_pretrained(new_model)
trainer.model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.save_pretrained(new_model)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

# Inference

In [20]:
# # Model
base_model_url = "/kaggle/input/llama-3.2/transformers/3b-instruct/1"
new_model_url = "senticXL/llama-3.2-3b-abstract-4"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
import torch
from trl import setup_chat_format

# Reload tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(base_model_url, padding_side='left')
base_model_reload= AutoModelForCausalLM.from_pretrained(
    base_model_url,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>'})
base_model_reload.resize_token_embeddings(len(tokenizer))

# Merge adapter with base model
tokenizer.chat_template = None
base_model_reload, tokenizer = setup_chat_format(base_model_reload, tokenizer)
model = PeftModel.from_pretrained(base_model_reload, new_model_url)

model = model.merge_and_unload()

In [ ]:
import transformers
pipeline = transformers.pipeline(
    "text-generation",
    tokenizer=tokenizer,
    model=model,
)

In [24]:
terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]
pipeline.tokenizer.pad_token_id = pipeline.model.config.eos_token_id

## POSULA

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files={"/kaggle/input/polusa/right_R_ELS.json"},split='train') 

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

batch_size = 64
decoded_outputs = []
dataloader = DataLoader(dataset, batch_size=batch_size)
for batch in tqdm(dataloader):
    output = pipeline(
        batch['text'],
        max_new_tokens=300,
        num_return_sequences=1,
        eos_token_id=terminators,
        do_sample=True,
        batch_size=batch_size
    )
    for idx, out in enumerate(output):
        text = out[0]['generated_text'].split('<|im_end|>')[-1]
        decoded_output = {'id':batch['id'][idx],'text': text.strip()}
        decoded_outputs.append(decoded_output)

In [28]:
import json
with open("right_R_ELS_abst.json", "w") as outfile:
    json.dump(decoded_outputs, outfile)

## Intergroup

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files={"/kaggle/input/intergroup/intergrop_test.json"},split='train') 

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

batch_size = 48
outputs = []
dataloader = DataLoader(dataset, batch_size=batch_size)
for batch in tqdm(dataloader):
    output = pipeline(
        batch['text'],
        max_new_tokens=256,
        num_return_sequences=1,
        eos_token_id=terminators,
        do_sample=True,
        batch_size=batch_size
    )
    outputs.extend(output)

In [ ]:
decoded_outputs = []
for idx, output in enumerate(outputs):
    text = output[0]['generated_text'].split('<|im_end|>')[-1]
    decoded_output = {
        'id': idx,
        'text': text
    }
    decoded_outputs.append(decoded_output)

In [ ]:
import json
with open("intergroup_test_outfile.json", "w") as outfile:
    json.dump(decoded_outputs, outfile)